# Norway vs Ivory Coast R32 prediction

Knockout prediction for Norway vs Ivory Coast (Cote d Ivoire in the data).

This notebook uses the updated local CSV inputs.

The model probabilities are 90-minute win/draw/loss probabilities. For knockout advancement, the draw bucket has to be allocated to extra time or penalties separately.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import poisson

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import load_data as L
from src.features import build_features as F
from src.models import logistic_model as LM
from src.models import poisson_model as P
from src.utils.config import RESULT_CLASSES

TEAM = "Norway"
OPPONENT = "Cote d Ivoire"
DISPLAY_OPPONENT = "Ivory Coast"
MATCH_DATE = "2026-06-30"
FIXTURE = [{"date": MATCH_DATE, "team": TEAM, "opponent": OPPONENT, "is_home": 1, "is_neutral": 1}]


## Load updated data

The fixture is created directly here because the repository's configured `FIXTURES` still describes the original group-stage matches.

In [2]:
matches = L.load_matches()
elo = L.load_elo()
fifa = L.load_fifa()
flashscore = L.load_flashscore()

persp_feat, model_ds, all_ds, fixture_ds = F.engineer(
    matches, elo, fifa, flashscore, fixtures=FIXTURE
)

fixture_row = fixture_ds[(fixture_ds["team"] == TEAM) & (fixture_ds["opponent"] == OPPONENT)].iloc[0]

print(f"matches: {len(matches):,}  |  Flashscore rows: {len(flashscore):,}")
print(f"fixture: {TEAM} vs {DISPLAY_OPPONENT} on {MATCH_DATE} (neutral)")
print(f"Elo before fixture: {TEAM} {fixture_row['team_elo_before']:.0f}, {DISPLAY_OPPONENT} {fixture_row['opponent_elo_before']:.0f}")
print(f"Elo gap: {fixture_row['elo_diff']:+.0f}")


23:08:00 | INFO    | src.preprocessing.load_data | loaded 4646 matches (2020-12-04 -> 2026-06-26)
23:08:01 | INFO    | src.preprocessing.load_data | loaded 514 Flashscore team-match rows (107 teams)
23:08:01 | INFO    | src.preprocessing.load_data | team-perspective rows: 9292 (4646 matches x 2 sides)
23:08:01 | INFO    | src.preprocessing.load_data | attached Flashscore stats (xg coverage: 191/9292 perspective rows)
23:08:15 | INFO    | src.features.build_features | model dataset: 295 focus rows | 9300 all-team rows | 1 fixture rows
matches: 4,646  |  Flashscore rows: 514
fixture: Norway vs Ivory Coast on 2026-06-30 (neutral)
Elo before fixture: Norway 1918, Ivory Coast 1743
Elo gap: +175


## Prediction

The ensemble is a simple average of the multinomial logistic model and the independent-Poisson score model.

In [3]:
fit = LM.fit_final(model_ds, train_override=all_ds)
log_model, features = fit["logistic"]

log_p = LM.proba_frame(log_model, fixture_ds.loc[[fixture_row.name], features]).iloc[0].to_dict()

mu, strengths = P.compute_strengths(persp_feat)
pois = P.predict(
    strengths,
    mu,
    fixture_row["team"],
    fixture_row["opponent"],
    int(fixture_row["is_home"]),
    int(fixture_row["is_neutral"]),
)
ens = {c: (log_p[c] + pois[c]) / 2 for c in RESULT_CLASSES}

summary = pd.DataFrame(
    {
        "Norway win": [log_p["win"], pois["win"], ens["win"]],
        "Draw": [log_p["draw"], pois["draw"], ens["draw"]],
        "Ivory Coast win": [log_p["loss"], pois["loss"], ens["loss"]],
    },
    index=["Logistic", "Poisson", "Ensemble"],
)

display((summary * 100).round(1).astype(str) + "%")

verdict = {"win": "Norway", "draw": "Draw after 90 minutes", "loss": DISPLAY_OPPONENT}[max(ens, key=ens.get)]
advance_norway = ens["win"] + 0.5 * ens["draw"]
advance_ivory = ens["loss"] + 0.5 * ens["draw"]

print(f"90-minute ensemble lean: {verdict}")
print(f"Expected goals: Norway {pois['lam_team']:.2f} | {DISPLAY_OPPONENT} {pois['lam_opp']:.2f}")
print(f"Most likely Poisson scoreline: Norway {pois['score'][0]}-{pois['score'][1]} {DISPLAY_OPPONENT}")
print(f"Naive advancement estimate if draw is split evenly: Norway {advance_norway:.1%}, {DISPLAY_OPPONENT} {advance_ivory:.1%}")


23:08:15 | INFO    | src.models.logistic_model | fitted logistic on 9300 rows (half-life=1095 d)
23:08:15 | INFO    | src.models.poisson_model | Poisson strengths: 201 teams, league avg = 1.32 goals/team/match


c:\Users\karoo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


,Norway win,Draw,Ivory Coast win
Logistic,45.6%,30.0%,24.3%
Poisson,33.2%,26.3%,40.5%
Ensemble,39.4%,28.2%,32.4%


90-minute ensemble lean: Norway
Expected goals: Norway 1.22 | Ivory Coast 1.38
Most likely Poisson scoreline: Norway 1-1 Ivory Coast
Naive advancement estimate if draw is split evenly: Norway 53.5%, Ivory Coast 46.5%


## Why it is close

Norway still has the Elo edge, but the newly-added match results pull recent form toward Ivory Coast.

In [4]:
feature_snapshot = pd.Series(
    {
        "elo_diff": fixture_row["elo_diff"],
        "rest_days_diff": fixture_row["rest_days_diff"],
        "win_rate_last_5_diff": fixture_row["win_rate_last_5_diff"],
        "points_per_game_last_5_diff": fixture_row["points_per_game_last_5_diff"],
        "goal_diff_avg_last_5_diff": fixture_row["goal_diff_avg_last_5_diff"],
        "goal_diff_avg_last_10_diff": fixture_row["goal_diff_avg_last_10_diff"],
        "Norway win_rate_last_5": fixture_row["team_win_rate_last_5"],
        "Ivory Coast win_rate_last_5": fixture_row["opponent_win_rate_last_5"],
        "Norway ppg_last_5": fixture_row["team_points_per_game_last_5"],
        "Ivory Coast ppg_last_5": fixture_row["opponent_points_per_game_last_5"],
        "Norway gd_last_5": fixture_row["team_goal_diff_avg_last_5"],
        "Ivory Coast gd_last_5": fixture_row["opponent_goal_diff_avg_last_5"],
    },
    name="value",
)
display(feature_snapshot.to_frame().round(3))

latest = matches[(matches["home_team"].isin([TEAM, OPPONENT])) | (matches["away_team"].isin([TEAM, OPPONENT]))]
latest = latest.sort_values("date")[["date", "home_team", "away_team", "home_score", "away_score", "competition_type"]]
display(latest.tail(12))


,value
elo_diff,175.000
rest_days_diff,-1.000
win_rate_last_5_diff,-0.071
points_per_game_last_5_diff,-0.143
goal_diff_avg_last_5_diff,-0.286
goal_diff_avg_last_10_diff,-0.042
Norway win_rate_last_5,0.643
Ivory Coast win_rate_last_5,0.714
Norway ppg_last_5,2.000
Ivory Coast ppg_last_5,2.143


,date,home_team,away_team,home_score,away_score,competition_type
4424,2026-03-28,Cote d Ivoire,South Korea,4,0,friendly
4448,2026-03-31,Norway,Switzerland,0,0,friendly
4469,2026-03-31,Cote d Ivoire,Scotland,1,0,friendly
4525,2026-06-01,Norway,Sweden,3,1,friendly
4543,2026-06-04,France,Cote d Ivoire,1,2,friendly
4594,2026-06-07,Morocco,Norway,1,1,friendly
4642,2026-06-15,Cote d Ivoire,Ecuador,1,0,world_cup
4638,2026-06-16,Iraq,Norway,1,4,world_cup
4643,2026-06-20,Germany,Cote d Ivoire,2,1,world_cup
4641,2026-06-23,Norway,Senegal,3,2,world_cup


## Scoreline grid

Independent Poisson scoreline probabilities, shown as the top five exact scores.

In [5]:
MAXG = 6
goals = np.arange(MAXG + 1)
a = poisson.pmf(goals, pois["lam_team"])
b = poisson.pmf(goals, pois["lam_opp"])
grid = np.outer(a, b)
grid = grid / grid.sum()

score_grid = pd.DataFrame(
    grid,
    index=pd.Index(range(MAXG + 1), name="Norway"),
    columns=pd.Index(range(MAXG + 1), name=DISPLAY_OPPONENT),
)
display((score_grid * 100).round(2))

flat = []
for i in range(MAXG + 1):
    for j in range(MAXG + 1):
        flat.append((f"{i}-{j}", grid[i, j]))
top5 = pd.DataFrame(flat, columns=["score", "probability"]).sort_values("probability", ascending=False).head(5)
display(top5.assign(probability=(top5["probability"] * 100).round(2).astype(str) + "%"))


Ivory Coast,0,1,2,3,4,5,6
Norway,,,,,,,
0,7.39,10.21,7.05,3.25,1.12,0.31,0.07
1,9.05,12.50,8.63,3.97,1.37,0.38,0.09
2,5.54,7.65,5.28,2.43,0.84,0.23,0.05
3,2.26,3.12,2.16,0.99,0.34,0.09,0.02
4,0.69,0.96,0.66,0.30,0.10,0.03,0.01
5,0.17,0.23,0.16,0.07,0.03,0.01,0.00
6,0.03,0.05,0.03,0.02,0.01,0.00,0.00


,score,probability
8,1-1,12.5%
1,0-1,10.21%
7,1-0,9.05%
9,1-2,8.63%
15,2-1,7.65%


## Current read

With the updated group results, this is best read as a near coin flip. The verified run gives an ensemble of Norway 39.4%, draw 28.2%, Ivory Coast 32.4% over 90 minutes. Poisson alone leans Ivory Coast, and the most likely scoreline is 1–1. If the draw bucket is split evenly for extra time / penalties, the rough advancement estimate is Norway 53.5% vs Ivory Coast 46.5%.